# Annotation quality: ground-truth source bias, and the vintage effect

Two tests on the recall ground truth. Both are free, in the sense that they need
no new annotation, and together with the agreement test and the false-negative
re-review they close the evaluation.

**1. OSM against manual annotation, at constant department.** The national recall
gap between OSM-sourced points (0.63) and manually annotated ones (0.54) is
ambiguous, and the ambiguity matters.

It could be *composition*: manual points are concentrated in the departments
where OSM coverage is thin, which may simply be harder. Or it could be genuine
*source bias*: OSM contributors map what is visible, so an OSM sample
over-represents detectable systems, and recall would be overstated wherever the
ground truth is mostly OSM.

The test separates the two. A logistic regression of the outcome on the source
with department fixed effects, restricted to departments where both sources are
present, isolates the source effect at constant composition. The national gap is
then decomposed into a composition part and a within-department part.

The direction of any bias is worth stating in advance: recall higher on OSM means
the composite recall is *overstated* where the ground truth is mostly OSM, which
understates the correction, which is conservative for an under-reporting claim.

**2. The vintage effect on pipeline recall.** The manuscript states that recall
does not depend on the year the imagery was taken. That claim was established on
the earlier *classification* recall and has to be re-established on the pipeline
recall, which is the quantity actually used.

**A caveat carried openly**: vintage and department are confounded, one campaign
per department, so identification is between departments and the test is
descriptive. The earlier analysis had the same limitation.

In [1]:
# Repository bootstrap: locate the root, then read every data location from paths.py.
import sys
from pathlib import Path


def _repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for parent in [start, *start.parents]:
        if (parent / "paths.py").exists() and (parent / "code").is_dir():
            return parent
    raise FileNotFoundError("run this notebook from inside the repository")


sys.path.insert(0, str(_repo_root()))
import paths


## 0. Chargement

In [2]:
import sys
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from scipy.stats import chi2_contingency

RECALL_POINTS = Path(paths.RECALL_POINTS)
DATES_FILE = Path(paths.IMAGERY_DATES)
MIN_PER_SOURCE = 30   # points required from EACH source for a department to enter test 1

rp = pd.read_csv(RECALL_POINTS, dtype={'dpt': str})
dates = gpd.read_file(DATES_FILE, columns=['dpt', 'imagedate'], ignore_geometry=True)
dates['imagedate'] = pd.to_datetime(dates['imagedate'], errors='coerce')
dept_year = dates.groupby('dpt')['imagedate'].agg(
    lambda x: int(x.dt.year.mode().iloc[0]) if x.notna().any() else np.nan)
rp['vintage'] = rp['dpt'].map(dept_year)
print(f"{len(rp)} points | sources: {rp['source'].value_counts().to_dict()} | "
      f"vintages: {rp['vintage'].value_counts().sort_index().to_dict()}")


13830 points | sources: {'OSM': 8441, 'manual_annotation': 5389} | vintages: {2022: 496, 2023: 2491, 2024: 5069, 2025: 5774}


## 1. OSM against manual annotation, at constant department

In [3]:
counts = rp.pivot_table(index='dpt', columns='source', values='pred', aggfunc='size').fillna(0)
eligible = counts[(counts.get('OSM', 0) >= MIN_PER_SOURCE) &
                  (counts.get('manual_annotation', 0) >= MIN_PER_SOURCE)].index
sub = rp[rp['dpt'].isin(eligible)].copy()
sub['is_osm'] = (sub['source'] == 'OSM').astype(int)
print(f"{len(eligible)} eligible departments (>= {MIN_PER_SOURCE} points from each source), "
      f"{len(sub)} points")

# Source effect at constant composition: logit on source with department fixed effects
fit = smf.logit('pred ~ is_osm + C(dpt)', data=sub).fit(disp=0)
coef, pval = fit.params['is_osm'], fit.pvalues['is_osm']
or_ = np.exp(coef)
print(f"\nsource coefficient (OSM vs manual), at constant department:")
print(f"  log-odds = {coef:+.3f} (OR = {or_:.2f}), p = {pval:.4f}")

# Decompose the national gap: raw against within-department
diff_nat = rp.loc[rp['source'] == 'OSM', 'pred'].mean() - \
           rp.loc[rp['source'] == 'manual_annotation', 'pred'].mean()
per_dept = sub.groupby(['dpt', 'source'])['pred'].mean().unstack()
diff_within = (per_dept['OSM'] - per_dept['manual_annotation'])
w = counts.loc[eligible].min(axis=1)   # weight by the limiting source
diff_within_w = np.average(diff_within, weights=w)
print(f"\nraw national gap (OSM - manual)            : {diff_nat:+.3f}")
print(f"within-department gap (weighted, eligible)   : {diff_within_w:+.3f}")
print(f"composition share ~ {diff_nat - diff_within_w:+.3f}")

if pval < 0.05:
    print("\n-> SOURCE BIAS is significant: the OSM sample over-represents detectable")
    print("   systems. Either weight the composite recall or carry it as a caveat. The")
    print("   direction (OSM recall above manual) means the composite recall is slightly")
    print("   OVERSTATED where the ground truth is mostly OSM, so the correction is")
    print("   understated, which is conservative for the under-reporting claim.")
else:
    print("\n-> No detectable source bias at constant composition: the national gap was")
    print("   a departmental composition effect. Nothing to change.")


16 eligible departments (>= 30 points from each source), 1875 points

source coefficient (OSM vs manual), at constant department:
  log-odds = -0.081 (OR = 0.92), p = 0.4293

raw national gap (OSM - manual)            : +0.094
within-department gap (weighted, eligible)   : -0.026
composition share ~ +0.120

-> No detectable source bias at constant composition: the national gap was
   a departmental composition effect. Nothing to change.


## 2. The vintage effect on pipeline recall

In [4]:
by_year = rp.groupby('vintage')['pred'].agg(['mean', 'count']).rename(columns={'mean': 'recall'})
display(by_year.round(3))

tab = pd.crosstab(rp['vintage'], rp['pred'])
chi2, p_chi2, dof, _ = chi2_contingency(tab)
print(f"chi2 = {chi2:.1f}, dof = {dof}, p = {p_chi2:.4f}  (raw test, geography confounded)")

# Same test controlling for local detection quality, through the department's F1
table_cal = pd.read_csv(paths.EVAL_TABLE, dtype={'dpt': str})
rp_f1 = rp.merge(table_cal[['dpt', 'f1']], on='dpt', how='left').dropna(subset=['f1', 'vintage'])
rp_f1['vintage'] = rp_f1['vintage'].astype(int)
fit0 = smf.logit('pred ~ f1', data=rp_f1).fit(disp=0)
fit1 = smf.logit('pred ~ f1 + C(vintage)', data=rp_f1).fit(disp=0)
lr = 2 * (fit1.llf - fit0.llf)
from scipy.stats import chi2 as chi2_dist
p_lr = 1 - chi2_dist.cdf(lr, df=fit1.df_model - fit0.df_model)
print(f"\nLR test on vintage (controlling for departmental F1): LR = {lr:.1f}, p = {p_lr:.4f}")
print("Caveat: vintage and department are confounded, one campaign per department,")
print("so identification is between departments and the reading is descriptive.")


,recall,count
vintage,,
2022,0.679,496
2023,0.576,2491
2024,0.580,5069
2025,0.611,5774


chi2 = 29.4, dof = 3, p = 0.0000  (raw test, geography confounded)

LR test on vintage (controlling for departmental F1): LR = 17.9, p = 0.0005
Caveat: vintage and department are confounded, one campaign per department,
so identification is between departments and the reading is descriptive.


## 3. What each test settles

Test 1 supports the ground-truth representativeness claim in the Methods, and
either resolves the spatial-bias limitation of an OSM-sourced ground truth or
quantifies it. Either outcome is usable; what would not be usable is leaving the
question open.

Test 2 is the one the manuscript's no-vintage-effect sentence should cite, since
it is measured on the pipeline recall rather than on classification recall, with
the confounding caveat stated.

Both write their results to `data/intermediate/evaluation/`.

In [5]:
pd.DataFrame({'metric': ['coef_source_logodds', 'or_source', 'p_source',
                          'diff_national', 'diff_within_weighted', 'n_depts_eligibles'],
              'value': [coef, or_, pval, diff_nat, diff_within_w, len(eligible)]}) \
    .to_csv(paths.SOURCE_CONCORDANCE, index=False)
by_year.assign(chi2_p=p_chi2, lr_p_controlling_f1=p_lr).to_csv(paths.VINTAGE_EFFECT)
print('Ecrit : source_concordance_results.csv, vintage_effect_results.csv')


Ecrit : source_concordance_results.csv, vintage_effect_results.csv
